# Capstone — mirrors your deployed research paper

This notebook holds the completed thinking and the code that backs the deployed paper (`docs/index.html`). Numbers below are transcribed from the completed `w04`–`w07` weekly notebooks, which contain the full working code against the anonymized dataset.

## 1. Question

**Research question:** Which content pages should be reviewed first for a possible refresh or optimisation, and does a learned model rank them better than a transparent hand-written rule?

**Decision this supports:** which pages a content editor or SEO analyst reviews first.

**Unit of analysis:** an individual content page.

**Output:** a ranked action score and reason code (a priority queue), not a single yes/no call.

**Cost of a wrong call:** editorial time spent on low-impact pages while a real decline/opportunity page goes unreviewed.

In [ ]:
decision = "Which content pages should be reviewed first for refresh or optimisation"
unit_of_analysis = "content page"
output = "ranked action score + reason code (priority queue)"

print("Decision:", decision)
print("Unit of analysis:", unit_of_analysis)
print("Output:", output)

## 2. Data

**Source:** FlyRank ML Internship dataset, Search Intelligence capstone release (`content_refresh_anonymized.csv`).

**Date window:** not established by the anonymized release used in the weekly notebooks — confirmed against the Hugging Face dataset card before final submission; no calendar window is invented here.

**Excluded:** client names, domains, URLs, private search queries, credentials — the release ships pre-anonymized with pseudonymous `client_id`/`content_id` only.

In [ ]:
dataset_rows = 30000
dataset_columns = 44
n_clients = 32

print(f"Rows: {dataset_rows:,}")
print(f"Columns: {dataset_columns}")
print(f"Pseudonymous clients: {n_clients}")

## 3. Methodology

**Baseline:** a transparent rule — staleness (`days_since_last_update > 180` → +40), search demand (`search_volume >= 100` → +30), ranking opportunity (`avg_position` 8–20 → +20), CTR opportunity (`ctr < 2` → +10). Outputs one of `STALE_REFRESH`, `QUICK_WIN`, `CTR_FIX`, `LOW_PRIORITY`.

**Model:** Logistic Regression (`max_iter=1000`), numeric features median-imputed and standardized, categorical features mode-imputed and one-hot encoded.

**Label:** `1` when `trend_direction == "down"`, else `0` — an observed historical outcome.

**Validation design:** grouped split by `client_id` (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) so no client appears in both sets.

**Leakage checks:** excluded `trend_direction`, `trend_pct` (define the label); `impressions_last_30d`, `impressions_prev_30d` (feed the trend calculation directly); and the baseline's own `score`/`reason_code`/`action_label` (decision outputs, not real-world inputs).

In [ ]:
excluded_features = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "score", "reason_code", "action_label",
]

split_params = {"method": "GroupShuffleSplit", "n_splits": 1, "test_size": 0.20, "random_state": 42}

print("Excluded (leakage) features:")
for f in excluded_features:
    print(" -", f)
print("\nSplit params:", split_params)

## 4. Results (vs baseline)

Both baseline and model were evaluated on the identical held-out client split, using the same Precision@50 metric.

In [ ]:
import pandas as pd

train_clients, test_clients = 25, 7
train_rows, test_rows = 23837, 6163
client_overlap = 0
test_base_rate = 0.511

results = pd.DataFrame({
    "method": ["Week-4 rule-based baseline", "Logistic Regression"],
    "precision_at_50": [0.540, 0.720],
    "test_base_rate": [test_base_rate, test_base_rate],
})

print(f"Train: {train_clients} clients / {train_rows:,} rows")
print(f"Test:  {test_clients} clients / {test_rows:,} rows")
print(f"Client overlap: {client_overlap}")
print(f"Top-50 false positives (model): 14\n")
display(results)

lift = results.loc[1, "precision_at_50"] - results.loc[0, "precision_at_50"]
print(f"Observed improvement: +{lift*100:.0f} percentage points on this split")

## 5. Limitations

- Correlational, not causal — coefficients and rule signals are associations, not mechanisms.
- Single split, single seed (`random_state=42`) — no cross-validated range reported.
- The label is an observed historical trend, not an independent ground truth of harm.
- Portfolio-specific: 30,000 pages across 32 clients may not generalise to a different mix.
- The model has 14 false positives in its top-50 — it produces a review queue, not an autonomous action system.
- No claim is made about predicting or reverse-engineering Google's ranking algorithm.

In [ ]:
claims_checklist = [
    "Results reported as observed and measured",
    "No causal claims made without experimental evidence",
    "No client-identifying information included",
    "Model compared fairly against baseline on the same validation split",
    "All reported results reproducible from the repository",
]

for c in claims_checklist:
    print("[x]", c)

## 6. Ranked recommendations

The action playbook exports every page as a ranked, reason-coded row for human review — highest score first. It is a queue for a person to work through, not an autonomous action engine: **model ranks → human reviews → editor decides action → outcome is monitored.**

In [ ]:
playbook = pd.DataFrame({
    "reason_code": ["STALE_REFRESH", "QUICK_WIN", "CTR_FIX", "LOW_PRIORITY"],
    "meaning": [
        "Old content with a refresh opportunity",
        "Ranking / search-demand opportunity",
        "Low CTR opportunity",
        "Does not meet stronger priority conditions",
    ],
    "recommended_action": [
        "Refresh Content",
        "Optimise Existing Content",
        "Improve Title & Meta",
        "Monitor",
    ],
})
display(playbook)

print("Full ranked queue: work/outputs/baseline_action_score.csv (30,000 rows)")

## 7. Artifacts the paper embeds

The two charts embedded in the deployed paper: Precision@50 (baseline vs. model) and the top model coefficients by absolute magnitude.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Precision@50 comparison
methods = ["Baseline", "Log. Regression"]
precisions = [0.54, 0.72]
axes[0].barh(methods, precisions, color=["#8B8D7F", "#1F6F63"])
axes[0].set_xlim(0, 1)
axes[0].set_title("Precision@50: baseline vs. model")
for i, v in enumerate(precisions):
    axes[0].text(v + 0.02, i, f"{v:.2f}", va="center")

# Top coefficients
features = [
    "users_90d", "sessions_90d", "days_with_impressions", "intent: navigational",
    "days_with_sessions", "type: keyword article", "type: feedly article",
    "content_age_days", "scroll_events_90d", "word_count",
]
coefs = [-0.988, 0.770, 0.604, -0.450, -0.427, 0.357, -0.340, -0.288, 0.281, 0.243]
colors = ["#1F6F63" if c < 0 else "#B5651D" for c in coefs]
axes[1].barh(features[::-1], coefs[::-1], color=colors[::-1])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Top Logistic Regression coefficients")

plt.tight_layout()
plt.savefig("capstone_charts.png", dpi=150)
plt.show()

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.